# 03 — Model Training

Train, evaluate, and compare the ML models in the `nexus_ml` ensemble.

**Models:**
- XGBoost direction classifier
- LightGBM regime detector
- Random Forest feature importance
- LSTM sequence predictor
- FinBERT news sentiment
- Isolation Forest anomaly detector

**Pipeline:**
1. Load features from DuckDB
2. Train/val/test split (walk-forward)
3. Train each model, log to MLflow
4. Ensemble voting & weighted average
5. Classification report + confusion matrix

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px

DB_PATH = "../data/nexus.duckdb"

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
features_df = conn.execute(
    "SELECT * FROM features WHERE symbol='EURUSD' AND timeframe='M15' ORDER BY timestamp"
).fetchdf()
conn.close()
print(f"Feature rows: {len(features_df)}")
features_df.head()

In [ ]:
# Create target: 1 if close[t+1] > close[t], else 0
if 'close' in features_df.columns:
    features_df['target'] = (features_df['close'].shift(-1) > features_df['close']).astype(int)
    features_df.dropna(inplace=True)

    feature_cols = [c for c in features_df.columns if c not in ('symbol', 'timeframe', 'timestamp', 'target', 'close', 'open', 'high', 'low', 'volume')]
    X = features_df[feature_cols].values
    y = features_df['target'].values
    print(f"Features: {len(feature_cols)}, Samples: {len(X)}, Class balance: {y.mean():.2%}")

In [ ]:
# XGBoost training
import xgboost as xgb

tscv = TimeSeriesSplit(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
    model.fit(X[train_idx], y[train_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=False)
    preds = model.predict(X[val_idx])
    print(f"Fold {fold}: Accuracy = {(preds == y[val_idx]).mean():.4f}")

In [ ]:
# Final evaluation
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

final_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
final_model.fit(X_train, y_train, verbose=False)
y_pred = final_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['DOWN', 'UP']))

In [ ]:
# Feature importance
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(20)
fig = px.bar(importance, orientation='h', title='Top 20 Feature Importances (XGBoost)')
fig.show()